# Module 1 — The Nature of Time Series Data
### Univariate Time Series Analysis · Recap with Python
> **Based on:** *Univariate Time Series Analysis*, SoSe 2024 (LMU Munich)  
> **Data sources:** `yfinance` (market data) · `fredapi` (FRED macro data) · Bundesbank via DBnomics  
> **Goal:** Recap core theory + see it applied to real trading/macro data in Python

---

## Contents
1. [What is a Time Series?](#1)
2. [Classical Decomposition](#2)
3. [Simple Filters — Moving Average & Hodrick-Prescott](#3)
4. [White Noise & the Random Walk](#4)
5. [Benchmark Forecast Models](#5)

---

## Setup — Install & Import
Run this cell once to install the required packages. After the first run you can comment out the `pip install` line.

In [ ]:
# !pip install yfinance fredapi statsmodels pandas matplotlib scipy pandas-datareader

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

import yfinance as yf
from fredapi import Fred
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf

# ── Plotting style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
TEAL   = '#1D9E75'
BLUE   = '#185FA5'
AMBER  = '#BA7517'
CORAL  = '#D85A30'
GRAY   = '#888780'

# ── FRED API key ───────────────────────────────────────────────────────────────
# Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html
FRED_KEY = 'YOUR_FRED_API_KEY_HERE'
fred = Fred(api_key=FRED_KEY)

print('All imports OK.')

<a id='1'></a>
---
## 1 · What is a Time Series?

### 📖 Theory

A **time series** is a temporally ordered sequence of observations:

$$\{y_1, y_2, \ldots, y_T\}$$

Unlike cross-sectional data, **order matters** — $y_3$ came before $y_4$, and the temporal distance carries information. The fundamental conceptual move in time series econometrics is to view the observed data as a single **realization** of an underlying **stochastic process**:

$$\{y_t\}_{t \in \mathbb{Z}}$$

A **stochastic process** is a family of random variables indexed by time $t$. Think of it as a machine that, at each point in time, draws a value from a probability distribution — possibly dependent on past draws. The dataset you have is just *one* path this machine happened to take.

### 🔑 Key Wohlrabe checklist before touching any series

| Question | Why it matters for trading |
|---|---|
| What frequency is the data? | Daily stock prices ≠ monthly macro figures. Mixing them requires care. |
| Is it seasonally adjusted? | Unadjusted retail sales spike every December — not a signal, just a calendar effect. |
| Who compiled it, and are there revisions? | GDP is revised months later. Real-time trading works on *first release*. |
| Levels or returns? | Prices are typically non-stationary (unit roots); log-returns are (approximately) stationary. |

### 📈 Trading perspective

In finance, you will almost always work with **log-returns** rather than price levels:

$$r_t = \log P_t - \log P_{t-1} \approx \frac{P_t - P_{t-1}}{P_{t-1}}$$

Why? Because price levels are non-stationary (they drift upward indefinitely), while log-returns are approximately stationary and have the convenient additive property: multi-period returns = sum of single-period log-returns.

In [ ]:
# ── Download DAX index data ────────────────────────────────────────────────────
# The DAX is the same series Wohlrabe uses in the lecture (slides 28–29)
dax = yf.download('^GDAXI', start='1990-01-02', end='2024-12-31', progress=False)
dax_close = dax['Close'].squeeze()

# Log-returns: the standard transformation for financial series
dax_ret = np.log(dax_close).diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(dax_close, color=TEAL, lw=0.8)
axes[0].set_title('DAX — Price level (one realization of a stochastic process)')
axes[0].set_ylabel('Index points')

axes[1].plot(dax_ret, color=BLUE, lw=0.5, alpha=0.8)
axes[1].axhline(0, color=GRAY, lw=0.6)
axes[1].set_title('DAX — Log-returns  (volatility clustering already visible)')
axes[1].set_ylabel('Log-return')

plt.tight_layout()
plt.show()

print(f'Observations : {len(dax_close):,}')
print(f'Date range   : {dax_close.index[0].date()} → {dax_close.index[-1].date()}')
print(f'Return mean  : {dax_ret.mean():.6f}')
print(f'Return std   : {dax_ret.std():.4f}')

> **What you see:** The level panel shows a clear upward trend — a non-stationary series.  
> The returns panel shows *volatility clustering*: calm periods alternate with turbulent ones.  
> That clustering is exactly what ARCH/GARCH models (Module 7) are designed to capture.

In [ ]:
# ── Descriptive statistics of returns ─────────────────────────────────────────
# In trading, the first thing you check is the distribution of your return series.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram vs Normal
axes[0].hist(dax_ret, bins=120, density=True, color=BLUE, alpha=0.6, label='DAX returns')
x = np.linspace(dax_ret.min(), dax_ret.max(), 300)
axes[0].plot(x, stats.norm.pdf(x, dax_ret.mean(), dax_ret.std()),
             color=CORAL, lw=2, label='Normal fit')
axes[0].set_title('Return distribution vs Normal')
axes[0].legend()

# QQ-plot: fat tails → points deviate at the extremes
stats.probplot(dax_ret, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot — Fat tails evident (Module 7 motivation)')

plt.tight_layout()
plt.show()

# Jarque-Bera test for normality
jb_stat, jb_p = stats.jarque_bera(dax_ret)
kurt = dax_ret.kurt()
print(f'Excess kurtosis : {kurt:.2f}  (Normal = 0; fat tails → > 0)')
print(f'Jarque-Bera p   : {jb_p:.2e}  (reject normality if p < 0.05)')

> **Trading insight:** Positive excess kurtosis means extreme returns happen *more often* than a normal distribution predicts.  
> This is the "fat tails" problem — it means naive VaR estimates (which assume normality) *underestimate* crash risk.  
> This is why risk managers use GARCH-based VaR instead (Module 7).

<a id='2'></a>
---
## 2 · Classical Decomposition

### 📖 Theory

The **additive decomposition model** (Wohlrabe slide 43) breaks any time series into four components:

$$y_t = d_t + c_t + s_t + \varepsilon_t$$

| Component | Symbol | Nature | Description |
|---|---|---|---|
| Trend | $d_t$ | Deterministic | Long-run persistent drift |
| Cyclical | $c_t$ | Deterministic | Medium-term (business cycle) swings |
| Seasonal | $s_t$ | Deterministic | Calendar-driven repetition |
| Irregular | $\varepsilon_t$ | **Stochastic** | The residual noise — should be stationary |

When the seasonal amplitude **grows proportionally** with the level (common in economic series), use the **multiplicative form**:

$$y_t = d_t \cdot s_t \cdot \varepsilon_t$$

The fix: take logs first → the multiplicative model becomes additive: $\log y_t = \log d_t + \log s_t + \log \varepsilon_t$.

### Goal of decomposition
Extract $d_t$, $c_t$, $s_t$ so that the residual $\varepsilon_t = y_t - d_t - c_t - s_t$ is **stationary and ideally white noise** — only then is the regular ARMA machinery valid.

### 📈 Trading perspective
Macro traders (e.g., those trading EUR/USD around German economic releases) need to understand whether a data surprise is structural (trend), cyclical (business cycle position), or just seasonal noise. A retail sales beat in December is expected — not a signal.

In [ ]:
# ── German Industrial Production — classical decomposition ───────────────────
# Wohlrabe uses exactly this series in the lecture (slides 26–27)
# FRED code: DEUPROINDMISMEI (Germany, industrial production, not seasonally adjusted)

ip_de = fred.get_series('DEUPROINDMISMEI', observation_start='1991-01-01')
ip_de = ip_de.dropna()
ip_de.name = 'German Industrial Production'

# Multiplicative decomposition: seasonal effect is proportional to the level
decomp = seasonal_decompose(ip_de, model='multiplicative', period=12, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
components = [
    (ip_de,            'Original series',             BLUE),
    (decomp.trend,     'Trend  $d_t$',                TEAL),
    (decomp.seasonal,  'Seasonal component  $s_t$',   AMBER),
    (decomp.resid,     'Irregular residual  $\\varepsilon_t$  (target of ARMA)', CORAL),
]
for ax, (series, title, color) in zip(axes, components):
    ax.plot(series, color=color, lw=0.9)
    ax.set_title(title)
    if 'Irregular' in title:
        ax.axhline(1, color=GRAY, lw=0.6, ls='--')  # multiplicative: 1 = no residual

plt.suptitle('Germany — Industrial Production: Multiplicative Decomposition', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

resid = decomp.resid.dropna()
print(f'Residual mean : {resid.mean():.4f}  (ideal = 1.0 for multiplicative)')
print(f'Residual std  : {resid.std():.4f}')

> **What you see:**
> - **Trend** captures the long-run level of German industrial activity, including the 2009 GFC crash and 2020 COVID shock.
> - **Seasonal** shows the recurring pattern (e.g., August summer slowdown, December dip) — constant amplitude confirms multiplicative was the right choice.
> - **Irregular residual** is what remains: ideally it should look like stationary noise around 1.0. Spikes here represent genuine surprises (COVID, supply chain shocks).

In [ ]:
# ── Seasonal adjustment: the simple dummy regression approach ─────────────────
# Wohlrabe slide 97: regress on 12 monthly dummies, residuals = seasonally adjusted series
# This is the econometrician's version of what CENSUS X-13 does more sophisticatedly.

import statsmodels.api as sm

ip_df = ip_de.to_frame(name='ip')
ip_df['month'] = ip_df.index.month

# Monthly dummies (no constant, so all 12 months included)
dummies = pd.get_dummies(ip_df['month'], prefix='m', drop_first=False).astype(float)
dummies.index = ip_df.index

model = sm.OLS(ip_df['ip'], dummies).fit()

# Seasonally adjusted = residuals + mean (to preserve level)
ip_sa = model.resid + ip_de.mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ip_de, color=BLUE, alpha=0.5, lw=0.9, label='Original (NSA)')
ax.plot(ip_sa, color=CORAL, lw=1.2, label='Seasonally adjusted (dummy method)')
ax.set_title('Germany — Industrial Production: Seasonal Adjustment via Monthly Dummies')
ax.legend()
plt.tight_layout()
plt.show()

print('Monthly seasonal factors (coefficients = average level by month):')
seasonal_factors = model.params
seasonal_factors.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(seasonal_factors.round(1).to_string())

<a id='3'></a>
---
## 3 · Simple Filters — Moving Average & Hodrick-Prescott

### 📖 Theory

Before fitting an ARMA model, we often need to isolate the trend. Two standard tools:

#### 3.1 Moving Average filter

$$\hat{y}_t^* = \frac{1}{L} \sum_{i=0}^{L-1} y_{t-i}$$

**Rules of thumb (Wohlrabe slide 48):**
- Window length $L$ should **exceed the cycle length** to smooth out the cyclical component
- $L$ should be **odd** (symmetric) to avoid introducing an artificial cyclical component
- Larger $L$ → smoother trend, but more end-point data loss

#### 3.2 Hodrick-Prescott (HP) Filter

The HP filter finds the trend $\tau_t$ that minimizes:

$$\min_{\tau_t} \left\{ \sum_{t=1}^T (y_t - \tau_t)^2 + \lambda \sum_{t=2}^{T-1} [(\tau_{t+1} - \tau_t) - (\tau_t - \tau_{t-1})]^2 \right\}$$

- **First term:** fit — the trend should track the data
- **Second term:** smoothness — the trend should not wiggle too much
- **$\lambda$** controls the trade-off:

| Frequency | Standard $\lambda$ |
|---|---|
| Quarterly | 1,600 |
| Monthly | 14,400 |
| Annual | 100 |

The **cyclical component** is then: $c_t = y_t - \tau_t$

### 📈 Trading perspective
The HP filter is used by central banks (ECB, Bundesbank) to estimate the **output gap** ($c_t$ above) — the deviation of GDP from potential. A large positive output gap → overheating → tightening bias → bond market implications.

In [ ]:
# ── German GDP: HP filter to extract trend and output gap ────────────────────
# FRED code: CLVMNACSCAB1GQDE = Germany, Real GDP, quarterly, chain-linked

gdp_de = fred.get_series('CLVMNACSCAB1GQDE', observation_start='1991-01-01')
gdp_de = gdp_de.dropna()

# HP filter — λ=1600 is the standard for quarterly data
gdp_cycle, gdp_trend = hpfilter(gdp_de, lamb=1600)

# Also compute a simple 5-quarter centred MA for comparison
gdp_ma = gdp_de.rolling(window=5, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(gdp_de,    color=BLUE,  lw=0.9, alpha=0.6, label='Real GDP (quarterly)')
axes[0].plot(gdp_trend, color=CORAL, lw=2.0, label='HP trend  (λ=1600)')
axes[0].plot(gdp_ma,    color=TEAL,  lw=1.5, ls='--', label='5-quarter centred MA')
axes[0].set_title('Germany — Real GDP and trend estimates')
axes[0].set_ylabel('Chain-linked index')
axes[0].legend()

axes[1].fill_between(gdp_cycle.index, gdp_cycle, 0,
                     where=gdp_cycle >= 0, alpha=0.5, color=TEAL,  label='Above trend (boom)')
axes[1].fill_between(gdp_cycle.index, gdp_cycle, 0,
                     where=gdp_cycle <  0, alpha=0.5, color=CORAL, label='Below trend (recession)')
axes[1].axhline(0, color=GRAY, lw=0.7)
axes[1].set_title('Output gap  $c_t = y_t - \\hat{\\tau}_t$  (HP cyclical component)')
axes[1].set_ylabel('Deviation from trend')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Sensitivity to λ: how much does the choice matter? ────────────────────────
# A key criticism of HP: the output gap estimate is sensitive to λ.
# Traders and policy analysts should always check robustness.

lambdas = [100, 400, 1600, 6400, 25600]
colors  = [BLUE, TEAL, CORAL, AMBER, GRAY]

fig, ax = plt.subplots(figsize=(13, 4))

for lam, col in zip(lambdas, colors):
    cyc, _ = hpfilter(gdp_de, lamb=lam)
    ax.plot(cyc, color=col, lw=1.2, alpha=0.8, label=f'λ = {lam:,}')

ax.axhline(0, color=GRAY, lw=0.5)
ax.set_title('Output gap sensitivity to λ — higher λ → smoother trend → larger apparent cycle')
ax.legend(ncol=5)
plt.tight_layout()
plt.show()

print('Key lesson: The HP filter is atheoretical — the output gap you get depends heavily')
print('on λ. Central banks are aware of this; they use it alongside other methods.')

<a id='4'></a>
---
## 4 · White Noise & the Random Walk

### 📖 Theory

These are the two most fundamental processes in time series analysis.

#### 4.1 White Noise

A process $\{\varepsilon_t\}$ is called **white noise** — written $\{\varepsilon_t\} \sim WN(0, \sigma^2)$ — if:

$$E(\varepsilon_t) = 0 \qquad \text{(zero mean)}$$
$$\gamma(0) = \text{Var}(\varepsilon_t) = \sigma^2 \qquad \text{(constant variance)}$$
$$\gamma(h) = \text{Cov}(\varepsilon_t, \varepsilon_{t-h}) = 0 \quad \forall \, h \neq 0 \qquad \text{(no autocorrelation)}$$

White noise is the ideal residual — what should be left after all structure has been modeled.
- **iid WN** (independent): the strongest form; each draw is entirely independent
- **Weak WN**: uncorrelated, but not necessarily independent (e.g., ARCH processes are WN but not iid)

#### 4.2 The Random Walk

$$y_t = y_{t-1} + \varepsilon_t \qquad \varepsilon_t \sim WN(0, \sigma^2)$$

By recursive substitution: $y_t = y_0 + \sum_{j=1}^t \varepsilon_j$

**Key property:** $\text{Var}(y_t) = t \cdot \sigma^2$ — grows without bound. This means the random walk is **non-stationary**.  
The best forecast of $y_{t+h}$ given information up to $t$ is simply $y_t$ (current level).

This is the **Efficient Market Hypothesis** in its weakest form: if markets are efficient, price changes are unforecastable, and prices follow (approximately) a random walk.

### 📈 Trading perspective
- If prices were a pure random walk, **no technical analysis would work**. The empirical question is whether returns have *any* predictable structure — and that's what ARMA modeling is about.
- For macro series (interest rates, FX), the random walk is often the **hardest benchmark to beat** in short-horizon forecasts (Meese-Rogoff puzzle for exchange rates).

In [ ]:
# ── Simulate: White Noise vs Random Walk ──────────────────────────────────────
# Replication of Wohlrabe's 'Fun with Random Walks' exercise (slide 118)

np.random.seed(42)
T   = 250
N   = 50   # number of random walk paths

wn      = np.random.normal(0, 1, T)
rw_all  = np.cumsum(np.random.normal(0, 1, (T, N)), axis=0)
rw_one  = rw_all[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: White noise
axes[0].plot(wn, color=TEAL, lw=0.8)
axes[0].axhline(0, color=GRAY, lw=0.5, ls='--')
axes[0].set_title('White noise  $\\varepsilon_t \\sim N(0,1)$')
axes[0].set_xlabel('t')

# Panel 2: 50 random walks — same starting point, wildly different paths
for i in range(N):
    axes[1].plot(rw_all[:, i], lw=0.4, alpha=0.35, color=BLUE)
axes[1].set_title(f'{N} random walks: same start, diverging paths')
axes[1].set_xlabel('t')

# Panel 3: Variance grows linearly with t
var_theoretical = np.arange(1, T+1) * 1.0   # σ²=1, Var(y_t) = t·σ²
var_empirical   = np.var(rw_all, axis=1)
axes[2].plot(var_theoretical, color=CORAL, lw=2, label='Theoretical  $t\\sigma^2$')
axes[2].plot(var_empirical,   color=BLUE,  lw=1, ls='--', label='Empirical (50 paths)')
axes[2].set_title('Variance grows linearly with $t$')
axes[2].set_xlabel('t')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Is the DAX a random walk? — First visual check with ACF ──────────────────
# A random walk has all ACF(h) ≈ 1 in levels; returns should have ACF ≈ 0 at all lags.
# (Formal testing comes in Module 6 — ADF unit root test)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))

# Levels
plot_acf(dax_close, lags=40, ax=axes[0, 0], color=BLUE, title='ACF — DAX level')
plot_acf(dax_close.diff().dropna(), lags=40, ax=axes[0, 1], color=TEAL,
         title='ACF — DAX first differences (≈ returns)')

# Squared returns (for ARCH effects — preview of Module 7)
plot_acf(dax_ret**2, lags=40, ax=axes[1, 0], color=CORAL,
         title='ACF — DAX squared returns  (ARCH structure visible)')
plot_acf(np.abs(dax_ret), lags=40, ax=axes[1, 1], color=AMBER,
         title='ACF — |DAX returns|  (absolute returns: slow decay)')

plt.suptitle('Autocorrelation structure of DAX — levels vs returns', y=1.01)
plt.tight_layout()
plt.show()

print('Key observations:')
print('  - Level ACF decays extremely slowly → clear sign of non-stationarity (unit root)')
print('  - Return ACF ≈ 0 at all lags → returns close to white noise (hard to predict)')
print('  - Squared/absolute return ACF ≠ 0 → volatility IS predictable → GARCH (Module 7)')

<a id='5'></a>
---
## 5 · Benchmark Forecast Models

### 📖 Theory

Before fitting any sophisticated model, establish what a **naïve benchmark** achieves. A model that cannot beat a simple benchmark is worthless — at least in-sample fit is meaningless; only **out-of-sample** performance matters.

#### Three standard benchmarks

| Benchmark | Formula | When optimal |
|---|---|---|
| Naïve (random walk) | $\hat{y}_{t+h} = y_t$ | Series is I(1) — non-stationary |
| Historical mean | $\hat{y}_{t+h} = \bar{y}$ | Series is stationary, no dynamics |
| AR(p) | $\hat{y}_{t+h} = \hat{\alpha}_0 + \sum \hat{\alpha}_i y_{t+1-i}$ | Stationary series with autocorrelation |

**Wohlrabe's benchmark of choice:** the AR(p) model selected by AIC/BIC. Any more complex model must beat it.

#### Forecast evaluation metrics

$$\text{RMSE} = \sqrt{\frac{1}{H}\sum_{h=1}^H (y_{T+h} - \hat{y}_{T+h})^2}$$
$$\text{MAE} = \frac{1}{H}\sum_{h=1}^H |y_{T+h} - \hat{y}_{T+h}|$$
$$\text{MAPE} = \frac{1}{H}\sum_{h=1}^H \left|\frac{y_{T+h} - \hat{y}_{T+h}}{y_{T+h}}\right| \times 100$$

### 📈 Trading perspective
For a macro trader:
- Forecasting **inflation** matters for duration positioning in bonds
- Even a small RMSE improvement over the random walk can translate to profitable trades if it consistently predicts the *direction* of a move
- In practice, the **direction accuracy** (% of correct sign predictions) is often more important than RMSE

In [ ]:
# ── US CPI Inflation: AR(4) vs naïve benchmark ───────────────────────────────
# A classic macro forecasting exercise; direct parallel to Wohlrabe's case studies.

# Download CPI and compute YoY inflation
cpi_us = fred.get_series('CPIAUCSL', observation_start='1960-01-01')
infl   = (cpi_us.pct_change(12) * 100).dropna()
infl.name = 'US CPI Inflation (YoY %)'

# Train/test split — hold out last 24 months
split  = -24
train  = infl.iloc[:split]
test   = infl.iloc[split:]

# ── Benchmark 1: Naïve (random walk) ──
fc_naive = pd.Series([train.iloc[-1]] * len(test), index=test.index)

# ── Benchmark 2: Historical mean ──
fc_mean  = pd.Series([train.mean()]  * len(test), index=test.index)

# ── Benchmark 3: AR(4) — the Wohlrabe benchmark ──
ar4_model = AutoReg(train, lags=4, trend='c').fit()
fc_ar4    = ar4_model.forecast(steps=len(test))
fc_ar4.index = test.index

# ── Plot ──
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(infl.iloc[-48:], color=BLUE,  lw=1.2, label='Actual inflation')
ax.plot(fc_ar4,          color=TEAL,  lw=1.5, ls='-',  label='AR(4) forecast')
ax.plot(fc_naive,        color=CORAL, lw=1.2, ls='--', label='Naïve (RW) forecast')
ax.plot(fc_mean,         color=GRAY,  lw=1.0, ls=':',  label='Historical mean forecast')
ax.axvline(test.index[0], color=GRAY, lw=0.8, ls='--')
ax.text(test.index[0], ax.get_ylim()[0]*1.02, ' ← train | test →', fontsize=9, color=GRAY)
ax.set_title('US CPI Inflation — Out-of-sample forecasts: AR(4) vs benchmarks')
ax.set_ylabel('YoY % change')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Forecast evaluation metrics ───────────────────────────────────────────────

def rmse(actual, forecast):
    return np.sqrt(np.mean((actual.values - forecast.values)**2))

def mae(actual, forecast):
    return np.mean(np.abs(actual.values - forecast.values))

def direction_accuracy(actual, forecast):
    """% of periods where the forecast correctly predicts the direction of change."""
    actual_chg   = np.diff(actual.values)
    forecast_chg = np.diff(forecast.values)
    return np.mean(np.sign(actual_chg) == np.sign(forecast_chg)) * 100

models = {
    'Naïve (RW)':         fc_naive,
    'Historical mean':    fc_mean,
    'AR(4)':              fc_ar4,
}

print(f'{"Model":<20} {"RMSE":>8} {"MAE":>8} {"Direction %":>12}')
print('-' * 52)
for name, fc in models.items():
    r = rmse(test, fc)
    m = mae(test, fc)
    d = direction_accuracy(test, fc)
    print(f'{name:<20} {r:>8.3f} {m:>8.3f} {d:>11.1f}%')

print('\nAR(4) model summary:')
print(ar4_model.summary().tables[1])

> **Reading the results:**
> - **RMSE/MAE:** lower is better. The AR(4) should outperform naïve and mean for inflation.
> - **Direction accuracy:** for trading, getting the *sign* of the move right matters more than point accuracy. Even 55–60% direction accuracy can be profitable with good position sizing.
> - If AR(4) barely beats naïve, this is telling you the series is nearly a random walk at this horizon — no predictability to exploit.

In [ ]:
# ── Rolling window evaluation: how stable is AR(4) performance over time? ─────
# A static train/test split can be misleading. Rolling windows show whether the
# model's advantage is stable or concentrated in specific sub-periods.

window   = 120   # 10 years of training data
h        = 1     # 1-month ahead

rolling_errors_ar4   = []
rolling_errors_naive = []
dates = []

for i in range(window, len(infl) - h):
    train_roll = infl.iloc[i - window : i]
    actual     = infl.iloc[i + h - 1]

    # AR(4) forecast
    try:
        ar_roll = AutoReg(train_roll, lags=4, trend='c').fit()
        fc_roll = ar_roll.forecast(steps=h).iloc[-1]
    except Exception:
        continue

    # Naïve forecast
    naive_roll = train_roll.iloc[-1]

    rolling_errors_ar4.append((actual - fc_roll)**2)
    rolling_errors_naive.append((actual - naive_roll)**2)
    dates.append(infl.index[i])

err_df = pd.DataFrame({
    'AR(4) sq. error':   rolling_errors_ar4,
    'Naïve sq. error':   rolling_errors_naive,
}, index=dates)

# Smooth with 24-month rolling mean to see trend
smooth = err_df.rolling(24).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(smooth['AR(4) sq. error'],  color=TEAL,  lw=1.5, label='AR(4)')
ax.plot(smooth['Naïve sq. error'],  color=CORAL, lw=1.5, ls='--', label='Naïve')
ax.set_title('Rolling 1-month-ahead squared forecast error (24m smoothed) — US CPI inflation')
ax.set_ylabel('MSE')
ax.legend()
plt.tight_layout()
plt.show()

print('When AR(4) is above Naïve → the model underperforms → probably a structural break period.')
print('(e.g., post-2021 inflation surge was difficult for any backwards-looking model to forecast)')

---
## Module 1 — Summary

| Concept | Key formula | Trading relevance |
|---|---|---|
| Log-returns | $r_t = \log P_t - \log P_{t-1}$ | Stationary; additive over horizons |
| Decomposition | $y_t = d_t + c_t + s_t + \varepsilon_t$ | Isolate signal from seasonal noise |
| HP filter | $\min (\text{fit} + \lambda \cdot \text{smoothness})$ | Output gap → central bank reaction function |
| White noise | $E(\varepsilon_t)=0$, $\gamma(h)=0$ | The ideal model residual |
| Random walk | $y_t = y_{t-1} + \varepsilon_t$ | Default assumption for price levels; hardest benchmark |
| AR(p) benchmark | $y_t = \alpha_0 + \sum \alpha_i y_{t-i} + \varepsilon_t$ | First model to beat; use AIC/BIC to select $p$ |

---

## ➜ Next: Module 2 — Formal Foundations
**Stationarity · ACF & PACF · Ergodicity · Wold Decomposition**

Module 2 makes all of this rigorous: we define stationarity precisely, derive the autocovariance function, and prove why the ARMA representation is valid for any stationary process (Wold's theorem). Along the way, we'll compute ACF/PACF for DAX returns and German bond yields — the first step toward model identification in Module 4.

---
*Notebook by Claude · Based on Wohlrabe UTSA 2024 · For educational use*